In [ ]:
!pip install transformers
!pip install torch
!pip install datasets

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from sklearn.model_selection import train_test_split

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "roberta-base"
MAX_LEN = 192
BATCH_SIZE = 16
LR = 2e-5
EPOCHS = 10

# ---- 1) Load ----
train_path = "/teamspace/studios/this_studio/data/train_subtask1.csv"
df = pd.read_csv(train_path)

# Usa esattamente le tue colonne
df["text"] = df["text"].fillna("")

# split train/val (se non hai già un validation file)
train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class AffectDataset(Dataset):
    def __init__(self, df, is_test=False):
        self.texts = df["text"].tolist()
        self.is_test = is_test

        if not is_test:
            self.y = df[["valence", "arousal"]].values.astype(np.float32)
        else:
            self.y = None

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}

        if not self.is_test:
            item["labels"] = torch.tensor(self.y[idx])  # shape (2,)
        return item

train_ds = AffectDataset(train_df, is_test=False)
val_ds = AffectDataset(val_df, is_test=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

# ---- 2) Model: regression on 2 targets ----
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    problem_type="regression"
).to(DEVICE)

optimizer = AdamW(model.parameters(), lr=LR)

def evaluate(model, loader):
    model.eval()
    preds, golds = [], []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            out = model(**batch)
            p = out.logits.detach().cpu().numpy()
            y = batch["labels"].detach().cpu().numpy()
            preds.append(p)
            golds.append(y)

    preds = np.vstack(preds)
    golds = np.vstack(golds)

    mae = np.mean(np.abs(preds - golds), axis=0)  # [valence_mae, arousal_mae]
    mse = np.mean((preds - golds) ** 2, axis=0)
    return mae, mse

# ---- 3) Train ----
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0

    for batch in train_loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        out = model(**batch)
        loss = out.loss
        loss.backward()

        optimizer.step()
        optimizer.zero_grad()

        total_loss += loss.item()

    train_loss = total_loss / len(train_loader)
    mae, mse = evaluate(model, val_loader)

    print(
        f"Epoch {epoch} | train_loss={train_loss:.4f} | "
        f"MAE(valence,arousal)=({mae[0]:.3f},{mae[1]:.3f}) | "
        f"MSE(valence,arousal)=({mse[0]:.3f},{mse[1]:.3f})"
    )



Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
